In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================
# SPLIT EXTRACTED FRAMES INTO TRAIN / VAL
# Shuffles frames within each class before split
# Test set will be collected separately later
# ============================================

import os
import random
import shutil
from pathlib import Path

In [3]:
# -----------------------------
# PATHS
# -----------------------------
BASE_DIR = "/content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization"

EXTRACTED_FRAMES_DIR = os.path.join(BASE_DIR, "extracted_frames")
PROCESSED_IMAGES_DIR = os.path.join(BASE_DIR, "processed_images")

TRAIN_DIR = os.path.join(PROCESSED_IMAGES_DIR, "train")
VAL_DIR = os.path.join(PROCESSED_IMAGES_DIR, "val")

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

print("EXTRACTED_FRAMES_DIR:", EXTRACTED_FRAMES_DIR)
print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR:", VAL_DIR)

EXTRACTED_FRAMES_DIR: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/extracted_frames
TRAIN_DIR: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/processed_images/train
VAL_DIR: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/processed_images/val


In [4]:
# -----------------------------
# CLASS NAMES
# -----------------------------
class_names = [
    "floor1_hallway",
    "floor2_hallway",
    "floor3_hallway",
    "front_lobby",
    "laundry",
    "ccu_lounge",
    "floor1_elevator_landmark",
    "floor2_elevator_landmark",
    "floor3_elevator_landmark"
]

In [5]:
# -----------------------------
# SETTINGS
# -----------------------------
val_ratio = 0.2
random_seed = 26

image_extensions = {".jpg", ".jpeg", ".png"}
random.seed(random_seed)

In [6]:
def get_image_files(folder_path):
    return [
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if Path(f).suffix.lower() in image_extensions
    ]

In [7]:
# -----------------------------
# SPLIT FRAMES FOR EACH CLASS
# -----------------------------
summary = {}

for class_name in class_names:
    class_input_dir = os.path.join(EXTRACTED_FRAMES_DIR, class_name)
    class_train_dir = os.path.join(TRAIN_DIR, class_name)
    class_val_dir = os.path.join(VAL_DIR, class_name)

    os.makedirs(class_train_dir, exist_ok=True)
    os.makedirs(class_val_dir, exist_ok=True)

    if not os.path.exists(class_input_dir):
        print(f"[WARNING] Missing extracted frames folder: {class_input_dir}")
        summary[class_name] = {"total": 0, "train": 0, "val": 0}
        continue

    image_files = get_image_files(class_input_dir)

    # Shuffle before splitting
    random.shuffle(image_files)

    total_count = len(image_files)
    val_count = int(total_count * val_ratio)
    train_count = total_count - val_count

    train_files = image_files[:train_count]
    val_files = image_files[train_count:]

    # Copy files into train folder
    for src_path in train_files:
        dst_path = os.path.join(class_train_dir, os.path.basename(src_path))
        shutil.copy2(src_path, dst_path)

    # Copy files into val folder
    for src_path in val_files:
        dst_path = os.path.join(class_val_dir, os.path.basename(src_path))
        shutil.copy2(src_path, dst_path)

    summary[class_name] = {
        "total": total_count,
        "train": len(train_files),
        "val": len(val_files)
    }

print("\n=== TRAIN / VAL SPLIT SUMMARY ===")
for class_name, counts in summary.items():
    print(
        f"{class_name}: total={counts['total']}, "
        f"train={counts['train']}, val={counts['val']}"
    )


=== TRAIN / VAL SPLIT SUMMARY ===
floor1_hallway: total=663, train=531, val=132
floor2_hallway: total=1914, train=1532, val=382
floor3_hallway: total=1013, train=811, val=202
front_lobby: total=759, train=608, val=151
laundry: total=385, train=308, val=77
ccu_lounge: total=553, train=443, val=110
floor1_elevator_landmark: total=502, train=402, val=100
floor2_elevator_landmark: total=526, train=421, val=105
floor3_elevator_landmark: total=408, train=327, val=81


In [8]:
# -----------------------------
# FINAL COUNT CHECK
# -----------------------------
print("\n=== FINAL FOLDER COUNTS ===")

for split_name, split_dir in [("train", TRAIN_DIR), ("val", VAL_DIR)]:
    print(f"\n{split_name.upper()}:")
    for class_name in class_names:
        class_dir = os.path.join(split_dir, class_name)
        if os.path.exists(class_dir):
            count = len(get_image_files(class_dir))
            print(f"  {class_name}: {count}")
        else:
            print(f"  {class_name}: 0")


=== FINAL FOLDER COUNTS ===

TRAIN:
  floor1_hallway: 531
  floor2_hallway: 1532
  floor3_hallway: 811
  front_lobby: 608
  laundry: 308
  ccu_lounge: 443
  floor1_elevator_landmark: 402
  floor2_elevator_landmark: 421
  floor3_elevator_landmark: 327

VAL:
  floor1_hallway: 132
  floor2_hallway: 382
  floor3_hallway: 202
  front_lobby: 151
  laundry: 77
  ccu_lounge: 110
  floor1_elevator_landmark: 100
  floor2_elevator_landmark: 105
  floor3_elevator_landmark: 81
